# Estágio 2.2 — Modelo de retrieval (two-tower)

Treina um modelo de **duas torres** (two-tower) — o mesmo padrão de
arquitetura usado por sistemas de recomendação em produção (YouTube,
Spotify e outros publicaram variantes disso) — pra ranquear vídeos por
relevância pra cada usuário.

**Ideia central**: a torre de usuário e a torre de vídeo aprendem a
projetar usuário e vídeo no *mesmo espaço vetorial*, de forma que
similaridade de cosseno entre os dois vetores ≈ "quão bem esse vídeo
combina com esse usuário". Isso permite retrieval rápido: pré-computa o
vetor de todo o catálogo uma vez, e pra recomendar é só um produto de
matrizes.

⚠️ **Aviso importante**, lido antes de qualquer coisa: os dados de
interação usados aqui são **sintéticos** (gerados por simulação, não uso
real). Isso é intencional para prototipar a arquitetura, mas significa que
as métricas de "acurácia" deste notebook não representam qualidade de
recomendação real — ver a seção **Limitações conhecidas** mais abaixo, que
é tão importante quanto o treino em si.


In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder

import sys
sys.path.append("../scripts")
import registry

I0000 00:00:1787698741.405444  686685 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787698742.594515  686685 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787698746.091846  686685 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Dados de entrada

`videos_new.csv` é a saída do Estágio 2.1 (embeddings VideoMAE já
calculados). Os dados de **interação** (o que cada usuário assistiu,
curtiu, comentou) são simulados abaixo — não existe, ainda, uma fonte de
interações reais.


In [2]:
video_df = pd.read_csv("../videos_new.csv")

## Simulação de interações (dados sintéticos)

Gera um histórico de visualização plausível pra 15.000 usuários fictícios:
quantos vídeos cada um assistiu (distribuição geométrica — a maioria vê
pouco, poucos veem muito), quanto tempo assistiram, se curtiram/comentaram
(ligado à retenção — quem assiste mais tende a reagir mais).

Isso serve pra prototipar a arquitetura do modelo sem precisar esperar
volume real de uso — mas **não substitui dados reais** quando eles
existirem (ver "Próximos passos" no fim do notebook).


In [3]:
# Fixa a semente para garantir reproducibilidade
np.random.seed(42)

NUM_USUARIOS = 15000
video_ids = video_df["id"].values
duracoes = dict(zip(video_df["id"], video_df["duracao"]))

interacoes = []

for user_id in range(1, NUM_USUARIOS + 1):
    num_videos_vistos = int(np.random.geometric(p=0.4))
    num_videos_vistos = min(num_videos_vistos, len(video_ids))

    videos_assistidos = np.random.choice(
        video_ids, size=num_videos_vistos, replace=False
    )

    for v_id in videos_assistidos:
        duracao_total = duracoes[v_id]

        if np.random.rand() < 0.6:
            porcentagem_vista = np.random.uniform(0.1, 0.7)
        else:
            porcentagem_vista = np.random.uniform(0.75, 1.0)

        tempo_visualizacao = int(round(duracao_total * porcentagem_vista))
        tempo_visualizacao = max(1, min(tempo_visualizacao, duracao_total))

        proporcao = tempo_visualizacao / duracao_total

        if proporcao > 0.7:
            like = 1 if np.random.rand() < 0.45 else 0
            comentou = 1 if np.random.rand() < 0.12 else 0
        else:
            like = 1 if np.random.rand() < 0.05 else 0
            comentou = 1 if np.random.rand() < 0.01 else 0

        interacoes.append({
            "user_id": f"usr_{user_id:04d}",
            "video_id": v_id,
            "tempo_visualizacao": tempo_visualizacao,
            "duracao_total_video": duracao_total,
            "like": like,
            "comentou": comentou,
        })

user_interactions_df = pd.DataFrame(interacoes)
user_interactions_df = user_interactions_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total de interações geradas: {len(user_interactions_df)}")

Total de interações geradas: 37505


## Amostras negativas

Pra treinar um classificador binário (gostou/não gostou), precisa de
exemplos negativos — vídeos que o usuário **não** interagiu. Sorteia, pra
cada usuário, uma quantidade de vídeos não vistos igual à quantidade de
vídeos vistos (proporção 1:1).


In [4]:
todos_videos = set(video_ids)

print("Mapeando histórico de visualizações por usuário...")
historico_usuarios = user_interactions_df.groupby("user_id")["video_id"].apply(set).to_dict()

amostras_negativas = []
RATIO_NEGATIVOS = 1

print("Gerando amostras negativas...")
for user_id, videos_vistos in historico_usuarios.items():
    videos_nao_vistos = list(todos_videos - videos_vistos)

    if not videos_nao_vistos:
        continue

    num_positivos = len(videos_vistos)
    qtd_negativos = min(num_positivos * RATIO_NEGATIVOS, len(videos_nao_vistos))

    if qtd_negativos > 0:
        negativos_sorteados = np.random.choice(videos_nao_vistos, size=qtd_negativos, replace=False)

        for v_id in negativos_sorteados:
            duracao = duracoes[v_id]
            amostras_negativas.append({
                "user_id": user_id,
                "video_id": v_id,
                "tempo_visualizacao": 0,
                "duracao_total_video": duracao,
                "like": 0,
                "comentou": 0,
                "label": 0,
            })

user_interactions_df["label"] = 1
df_completo = pd.concat([user_interactions_df, pd.DataFrame(amostras_negativas)], ignore_index=True)
df_completo = df_completo.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total de amostras no dataset final: {len(df_completo)}")
print(df_completo["label"].value_counts())

Mapeando histórico de visualizações por usuário...
Gerando amostras negativas...
Total de amostras no dataset final: 75010
label
1    37505
0    37505
Name: count, dtype: int64


In [5]:
df = df_completo.merge(
    video_df.rename(columns={"id": "video_id"}),
    on="video_id",
)
df["taxa_retencao"] = df["tempo_visualizacao"] / df["duracao_total_video"]
df["sample_weight"] = (df["tempo_visualizacao"] * 3.0) + (df["like"] * 1.0) + (df["comentou"] * 1.0)
df.head()

,user_id,video_id,tempo_visualizacao,duracao_total_video,like,comentou,label,titulo,duracao,termo_busca,salvo_em,url,legenda_visual,tags_visao,embedding_videomae,taxa_retencao,sample_weight
0,usr_4488,MUDeMdHeudw,44,58,0,0,1,Você é Analista de Dados? Qual a sua média sal...,58,ciencia de dados,2026-08-16T21:37:58,https://www.youtube.com/watch?v=MUDeMdHeudw,The image is a collage of three photos. The fi...,"collage,three,photos,first,photo,left,person,s...",[ 2.47019455e-01 -5.57332560e-02 -2.63988785e-...,0.758621,132.0
1,usr_0917,Qx5OS0U6qJY,0,10,0,0,0,3 livros que todo empresário deveria ler #livr...,10,empreendedorismo,2026-08-19T16:44:48,https://www.youtube.com/watch?v=Qx5OS0U6qJY,The image is a collage of three different pict...,"collage,three,different,pictures,first,picture...",[ 2.76986152e-01 -7.45536909e-02 3.54749262e-...,0.000000,0.0
2,usr_6417,sDRuKmtW0Es,0,44,0,0,0,Análise de Dados | Exploratória versus Explana...,44,Analise exploratóoria,2026-08-19T16:42:39,https://www.youtube.com/watch?v=sDRuKmtW0Es,The image is a screenshot of a video call betw...,"screenshot,video,call,between,people,left,side...",[ 9.12219435e-02 -1.07489586e-01 4.49146191e-...,0.000000,0.0
3,usr_7817,wbT8L3Qzc6k,5,45,0,0,1,Scikit-Learn Library in Pyhton,45,biblioteca scikit-learn,2026-08-19T16:43:38,https://www.youtube.com/watch?v=wbT8L3Qzc6k,The image is a slide from a data science guide...,"slide,data,science,guide,blue,white,title,scik...",[-3.45721208e-02 -1.51040569e-01 -8.34560394e-...,0.111111,15.0
4,usr_1607,Oeh1xU4igJM,0,52,0,0,0,اهم #وظيفة ←#machinelearning,52,machine learning,2026-08-16T21:37:09,https://www.youtube.com/watch?v=Oeh1xU4igJM,The image is a screenshot from a video intervi...,"screenshot,video,interview,man,bald,head,glass...",[-6.16541393e-02 -8.50179791e-02 1.05396882e-...,0.000000,0.0


## Salvando as interações pra avaliação depois

Grava só o necessário pra medir Recall@K/NDCG@K (`scripts/avaliacao.py` e
`scripts/comparar_modelos.py`): `user_id`, `video_id`, `label`. Esse
arquivo fica fixo — é o que garante que toda versão do modelo (a sua e a
de quem contribuir) é avaliada exatamente pelo mesmo critério.


In [6]:
from pathlib import Path

Path("../data").mkdir(exist_ok=True)
df[["user_id", "video_id", "label"]].to_csv("../data/interacoes.csv", index=False)
print(f"Salvo: data/interacoes.csv ({len(df)} interações)")

Salvo: data/interacoes.csv (75010 interações)


## Preparando os embeddings pro modelo

Converte a coluna `embedding_videomae` (salva como string no CSV) de volta
pra array numérico, e monta a matriz `X_video_emb` (uma linha de 768
valores por interação).


In [7]:
def parse_embedding(x):
    if isinstance(x, str):
        return np.fromstring(x.strip("[]\n"), sep=" ")
    return np.array(x, dtype="float32")

video_df["embedding_videomae_parsed"] = video_df["embedding_videomae"].apply(parse_embedding)
video_emb_dict = dict(zip(video_df["id"], video_df["embedding_videomae_parsed"]))

df["embedding_videomae"] = df["embedding_videomae"].apply(parse_embedding)
X_video_emb = np.vstack(df["embedding_videomae"].values).astype("float32")

EMB_VIDEOMAE_DIM = X_video_emb.shape[1]
print(f"Formato da matriz VideoMAE: {X_video_emb.shape}")

Formato da matriz VideoMAE: (75010, 768)


## Arquitetura: torre de usuário

`user_id` → embedding aprendido (64d) → MLP (256 → 128) → normalização L2
(vetor unitário, pra que produto escalar = similaridade de cosseno).


In [15]:
n_user = df_completo["user_id"].nunique()
n_video = df_completo["video_id"].nunique()

u_id_input = layers.Input(shape=(1,), name="user_id")
u_id_embedding = layers.Embedding(input_dim=n_user + 1, output_dim=32, name="user_embedding_layer")(u_id_input)
u_id_flatten = layers.Flatten()(u_id_embedding)

x = layers.Dense(64, use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(u_id_flatten)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(32)(x)  # mesma dimensão da torre de vídeo
user_embedding = layers.UnitNormalization(name="user_embedding")(x)
user_tower = tf.keras.Model(inputs=u_id_input, outputs=user_embedding, name="User_Tower")

## Arquitetura: torre de vídeo

`video_id` (embedding aprendido, 64d) concatenado com o embedding VideoMAE
(768d, já treinado, vindo do Estágio 2.1) → mesma estrutura MLP → 128d
normalizado. É essa concatenação que faz o modelo enxergar tanto
"identidade do vídeo" quanto "conteúdo visual" do vídeo.


In [16]:
i_id_input = layers.Input(shape=(1,), name="video_id")
i_id_embedding = layers.Embedding(input_dim=n_video + 1, output_dim=32)(i_id_input)
i_id_flatten = layers.Flatten()(i_id_embedding)

v_dense_emb_vect = layers.Input(shape=(EMB_VIDEOMAE_DIM,), name="embedding_videomae")
video_feature = layers.Concatenate()([i_id_flatten, v_dense_emb_vect])

x = layers.Dense(64, use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(1e-4))(video_feature)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Dropout(0.1)(x)

x = layers.Dense(32)(x)
video_embedding = layers.UnitNormalization(name="video_embedding")(x)

video_tower = tf.keras.Model(
    inputs=[i_id_input, v_dense_emb_vect],
    outputs=video_embedding,
    name="Video_Tower",
)

## Combinando as duas torres

O produto escalar de dois vetores unitários é exatamente a similaridade de
cosseno. `TemperatureScale` aprende um fator de escala e um bias sobre esse
cosseno antes da sigmoid — uma técnica comum pra dar ao modelo mais
liberdade de calibrar a confiança das previsões.


In [17]:
class TemperatureScale(layers.Layer):
    """Multiplica o cosseno por um fator de escala (1/tau) e soma um bias treinável."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.scale = self.add_weight(name="scale", shape=(), initializer=tf.keras.initializers.Constant(10.0), trainable=True)
        self.bias = self.add_weight(name="bias", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        u_emb, v_emb = inputs
        cosine_sim = tf.reduce_sum(tf.multiply(u_emb, v_emb), axis=-1, keepdims=True)
        return (cosine_sim * self.scale) + self.bias


u_emb = user_tower(u_id_input)
v_emb = video_tower([i_id_input, v_dense_emb_vect])

logits = TemperatureScale()([u_emb, v_emb])
output = layers.Activation("sigmoid", name="pred_label")(logits)

training_model = tf.keras.Model(
    inputs={
        "user_id": u_id_input,
        "video_id": i_id_input,
        "embedding_videomae": v_dense_emb_vect,
    },
    outputs=output,
    name="Two_Tower_Training_Model",
)

training_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["AUC", "Precision", "Recall"],
)

In [18]:
user_encoder = LabelEncoder()
video_encoder = LabelEncoder()

user_ids_enc = user_encoder.fit_transform(df["user_id"]).astype("int32")
video_ids_enc = video_encoder.fit_transform(df["video_id"]).astype("int32")
labels = df["label"].astype("float32").values

n_user = len(user_encoder.classes_)
n_video = len(video_encoder.classes_)

history = training_model.fit(
    x={
        "user_id": user_ids_enc,
        "video_id": video_ids_enc,
        "embedding_videomae": X_video_emb,
    },
    y=labels,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
)

Epoch 1/10


W0000 00:00:1787699097.167459  686685 cpu_allocator_impl.cc:82] Allocation of 184344576 exceeds 10% of free system memory.


232/235 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - AUC: 0.4919 - Precision: 0.4913 - Recall: 0.4886 - loss: 0.7855

W0000 00:00:1787699105.010049  686685 cpu_allocator_impl.cc:82] Allocation of 46086144 exceeds 10% of free system memory.


235/235 ━━━━━━━━━━━━━━━━━━━━ 8s 19ms/step - AUC: 0.4921 - Precision: 0.4914 - Recall: 0.4873 - loss: 0.7849 - val_AUC: 0.4959 - val_Precision: 0.5062 - val_Recall: 0.3053 - val_loss: 0.7216
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - AUC: 0.6989 - Precision: 0.6435 - Recall: 0.6428 - loss: 0.6467 - val_AUC: 0.4977 - val_Precision: 0.5032 - val_Recall: 0.4194 - val_loss: 0.7644
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - AUC: 0.8230 - Precision: 0.7474 - Recall: 0.7407 - loss: 0.5330 - val_AUC: 0.4976 - val_Precision: 0.5064 - val_Recall: 0.4798 - val_loss: 0.8757
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - AUC: 0.9067 - Precision: 0.8244 - Recall: 0.8239 - loss: 0.4049 - val_AUC: 0.4959 - val_Precision: 0.5080 - val_Recall: 0.5098 - val_loss: 1.0166
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - AUC: 0.9476 - Precision: 0.8740 - Recall: 0.8740 - loss: 0.3156 - val_AUC: 0.4965 - val_Precision: 0.5058 - val_Recall: 0.5001 - val_loss: 1.1325
Epoch 

## Limitações conhecidas — leia antes de usar isso pra qualquer coisa séria

Observando o treino: **AUC de treino chega a ~0,999 já na última época,
enquanto AUC de validação fica travada em ~0,50–0,51** — ou seja, no
conjunto de validação o modelo acerta tanto quanto jogar uma moeda.

**Por que isso acontece:**

1. **Dados 100% sintéticos.** As interações foram geradas por regras
   aleatórias simples (`np.random`), sem nenhum padrão real de gosto por
   trás. Não existe sinal genuíno de "que tipo de vídeo esse usuário
   prefere" pra o modelo aprender — só ruído.
2. **Memorização de IDs, não generalização de gosto.** As torres de
   usuário e vídeo aprendem embeddings por `user_id`/`video_id`
   individuais. Com poucos exemplos por usuário e nenhum padrão real, o
   modelo memoriza os pares vistos no treino em vez de aprender uma
   função que generaliza — clássico overfitting.
3. **Catálogo pequeno (389 vídeos).** Poucos itens tornam mais fácil
   decorar do que generalizar.

**O que isso significa na prática**: esse modelo, do jeito que está, não
deve ser interpretado como "aprendeu a recomendar bem" — é uma prova de
arquitetura (a estrutura two-tower está correta e funcional), não uma
prova de qualidade de recomendação.

**Próximos passos pra evoluir isso de verdade** (ver também o README
deste estágio):
- Substituir as interações sintéticas por dados reais de uso — o Estágio 3
  já grava curtidas, "não gostei" e tempo de visualização por usuário
  (`03-serving-streaming`, tabela `events`), que é exatamente esse
  substituto.
- Adicionar uma métrica de retrieval de verdade (Recall@K, NDCG) em vez de
  só AUC de classificação — mais fiel ao que o modelo realmente faz em
  produção (ranquear, não classificar).
- Regularização mais forte e/ou redução da capacidade dos embeddings de ID
  pra dificultar a memorização.


## Pré-computando o catálogo e testando o retrieval

Roda a torre de vídeo uma vez sobre todo o catálogo — é essa matriz
pré-computada que permite recomendar em tempo real sem reprocessar nada
pesado a cada requisição.


In [19]:
videos_unicos_ids = video_df["id"].values
video_ids_encoded = video_encoder.transform(videos_unicos_ids).astype("int32")

X_video_mae_catalog = np.array([video_emb_dict[vid] for vid in videos_unicos_ids], dtype="float32")

catalog_video_embeddings = video_tower.predict(
    [video_ids_encoded, X_video_mae_catalog],
    batch_size=256,
)

print(f"Base vetorial criada: {catalog_video_embeddings.shape}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
Base vetorial criada: (389, 32)


In [20]:
def recomendar_videos(user_id_str, top_k=10):
    """Recebe o ID do usuário, gera o vetor via User Tower, calcula o
    produto escalar contra o catálogo e retorna os Top-K vídeos."""
    user_idx = user_encoder.transform([user_id_str]).astype("int32")
    user_emb = user_tower.predict(user_idx, verbose=0)

    scores = np.dot(catalog_video_embeddings, user_emb.T).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]

    return pd.DataFrame([
        {"video_id": videos_unicos_ids[idx], "score_cosseno": float(scores[idx])}
        for idx in top_indices
    ])


# --- Exemplo de uso ---
recomendar_videos(user_id_str="usr_11295", top_k=10)

,video_id,score_cosseno
0,mk1ogVlRuvs,0.628362
1,15g0Zl64NwY,0.540514
2,q9KSunjUmCA,0.514059
3,GDkVq42LN7w,0.499616
4,Ko59gOqx0K4,0.484225
5,kz7Q0zsWrds,0.474257
6,N1wNgWA0sVc,0.457338
7,Y6hl61l3wtI,0.450304
8,tvdQJ7r0OEk,0.447881
9,mvMMvC4mdKk,0.445984


## Salvando esta versão do modelo

Salva as duas torres, o catálogo pré-computado e as métricas de treino
numa pasta versionada — **sem promover automaticamente a produção**.
Depois de conferir as métricas, promova explicitamente com
`scripts/promover_modelo.py`.


In [21]:
metricas = {
    "train_auc": float(history.history["AUC"][-1]),
    "val_auc": float(history.history["val_AUC"][-1]),
    "epochs": len(history.history["AUC"]),
}

# pré-computa o vetor de TODO usuário conhecido (não só do catálogo) —
# necessário pra avaliação (scripts/avaliacao.py) e pra um dia servir o
# modelo em produção via lookup direto, sem rodar TensorFlow por requisição
usuarios_unicos_ids = user_encoder.classes_
usuarios_ids_encoded = np.arange(len(usuarios_unicos_ids)).astype("int32")
todos_usuarios_embeddings = user_tower.predict(usuarios_ids_encoded, batch_size=256)
print(f"Vetores de usuário pré-computados: {todos_usuarios_embeddings.shape}")

versao = registry.salvar_versao(
    user_tower=user_tower,
    video_tower=video_tower,
    catalog_embeddings=catalog_video_embeddings,
    catalog_video_ids=videos_unicos_ids,
    user_embeddings=todos_usuarios_embeddings,
    user_ids=usuarios_unicos_ids,
    metricas=metricas,
    notas="Dados de interação sintéticos. Overfitting severo (val AUC ~0.50) — ver seção de limitações acima.",
)

59/59 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Vetores de usuário pré-computados: (15000, 32)
Versão 'v_2026-08-26_000545_0036' salva em /home/julio/Documentos/documents/video-recsys-pipeline/02-embeddings-retrieval/models/two_tower/v_2026-08-26_000545_0036
Ainda não é a versão ativa (atual: v_2026-08-26_000316_302b).
Pra promover: python scripts/promover_modelo.py v_2026-08-26_000545_0036


## Como treinar uma versão nova e substituir esta

1. Rode este notebook de novo (idealmente já com dados de interação reais,
   não os sintéticos) — a célula acima cria uma versão nova automaticamente,
   sem apagar a anterior.
2. Compare as métricas das versões: `python scripts/promover_modelo.py --listar`
3. Gostou da versão nova? Promova: `python scripts/promover_modelo.py <nome_da_versao>`
4. Pronto — qualquer código que use `registry.carregar_modelo_atual()`
   (este notebook, ou uma futura API de recomendação) passa a usar o
   modelo novo automaticamente, sem precisar mudar nada além do registro.


## Validando se essa versão é melhor (pra contribuições)

```bash
python scripts/comparar_modelos.py {nome_da_versao_que_apareceu_acima}
```

Isso calcula Recall@10 e NDCG@10 da versão nova contra a versão que está
"atual" no registro, usando o holdout de `data/interacoes.csv`. Só depois
de ver um resultado igual ou melhor é que faz sentido promover — ver
`CONTRIBUTING.md` na raiz do repositório pra o fluxo completo de
contribuição (é assim que o CI valida automaticamente qualquer modelo
que alguém propuser via pull request).
